In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
!pip install fair-esm

In [ ]:
!pip install biopython

In [ ]:
import torch
import esm
import os
import numpy as np
import pandas as pd
from Bio import PDB

# Base path for the folds
base_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine'
new_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine'

In [ ]:
# Load the pre-trained ESM-2 model and its alphabet
esm2_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
esm2_model.eval()  # Disables dropout for deterministic results

In [ ]:
def get_esm2_embedding(sequence, model, alphabet):
    batch_converter = alphabet.get_batch_converter()
    data = [("protein", sequence)]
    batch_labels, batch_strs, batch_tokens = batch_converter(data)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[6], return_contacts=True)
    token_representations = results["representations"][6]

    sequence_embedding = token_representations[0, 1 : batch_lens[0] - 1]
    return sequence_embedding

In [ ]:
#@title Get antigen surface index tsv files from antigen_pdb_with_index

from Bio import PDB
import pandas as pd
import os

# Initialize PDB Parser
parser = PDB.PDBParser()

# Function to calculate average B-factor for a residue
def average_b_factor(residue):
    b_factors = [atom.get_bfactor() for atom in residue]
    return sum(b_factors) / len(b_factors) if b_factors else None


pdb_input_dir = os.path.join(base_fold_path, 'data', 'antigen_pdbs')
tsv_output_dir = os.path.join(base_fold_path, 'data', 'antigen_info_SI')

os.makedirs(tsv_output_dir, exist_ok=True)

# Process each PDB file in the input directory
for pdb_filename in os.listdir(pdb_input_dir):
    if pdb_filename.endswith('.pdb'):
        pdb_filepath = os.path.join(pdb_input_dir, pdb_filename)
        # Extract the PDB ID and the chain part from the filename, excluding '_antigen_updated'
        structure_id_parts = pdb_filename.split('_')[:-2]
        structure_id = "_".join(structure_id_parts)
        structure = parser.get_structure(structure_id, pdb_filepath)

        # List to hold the data
        data = []

        # Iterate over all residues in the structure
        for model in structure:
            for chain in model:
                for residue in chain:
                    if PDB.is_aa(residue, standard=True):  # Check if it's a standard amino acid
                        # Get the residue ID as a tuple (hetatm flag, residue sequence number, insertion code)
                        res_id_tuple = residue.get_id()
                        # Combine tuple elements to create a unique residue ID string
                        res_id = f"{res_id_tuple[0]}{res_id_tuple[1]}{res_id_tuple[2]}".strip()
                        res_name = residue.resname
                        si = average_b_factor(residue)
                        # Append the residue info to the data list
                        data.append([structure_id, res_id, res_name, si])

        # Convert the data into a pandas DataFrame
        df = pd.DataFrame(data, columns=['name', 'ResID', 'AA', 'SI'])
        # Save the DataFrame to a TSV file
        structure_id = pdb_filename.replace('.pdb', '')  # 全名不截断
        parts = structure_id.split('_')
        trimmed_id = '_'.join(parts[:-1])  # 去掉最后一个片段
        tsv_filename = f"{trimmed_id}_info.tsv"
        tsv_filepath = os.path.join(tsv_output_dir, tsv_filename)
        df.to_csv(tsv_filepath, sep='\t', index=False)
        print(f"Processed {pdb_filename} and saved to {tsv_filename}")

In [ ]:
si_input_dir = os.path.join(base_fold_path, 'data', 'antigen_info_SI')
interaction_input_dir = os.path.join(base_fold_path, 'data', 'antigen_info')
output_dir = os.path.join(base_fold_path, 'data', 'antigen_info_SI_interaction')

os.makedirs(output_dir, exist_ok=True)

# Process each TSV file in the SI input directory
for tsv_filename in os.listdir(si_input_dir):
    si_filepath = os.path.join(si_input_dir, tsv_filename)
    interaction_filepath = os.path.join(interaction_input_dir, tsv_filename)

    # Check if the corresponding interaction file exists
    if os.path.exists(interaction_filepath):
        # Read both files into dataframes
        si_df = pd.read_csv(si_filepath, sep='\t')
        interaction_df = pd.read_csv(interaction_filepath, sep='\t')

        # Ensure 'ResID' and 'AA' are of type string for both dataframes
        si_df['ResID'] = si_df['ResID'].astype(str)
        interaction_df['ResID'] = interaction_df['ResID'].astype(str)
        si_df['AA'] = si_df['AA'].astype(str)
        interaction_df['AA'] = interaction_df['AA'].astype(str)

        # Merge the dataframes on 'ResID' and 'AA'
        merged_df = pd.merge(interaction_df, si_df[['ResID', 'AA', 'SI']], on=['ResID', 'AA'], how='left')

        # Save the merged dataframe to a new TSV file in the output directory
        output_filepath = os.path.join(output_dir, tsv_filename)
        merged_df.to_csv(output_filepath, sep='\t', index=False)
        print(f"Merged file saved to {output_filepath}")
    else:
        print(f"No corresponding interaction file found for {tsv_filename}.")

In [ ]:
# Map three-letter amino acid codes to one-letter codes
aa_map = {
    'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D', 'CYS': 'C',
    'GLU': 'E', 'GLN': 'Q', 'GLY': 'G', 'HIS': 'H', 'ILE': 'I',
    'LEU': 'L', 'LYS': 'K', 'MET': 'M', 'PHE': 'F', 'PRO': 'P',
    'SER': 'S', 'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'
}

input_dir = os.path.join(base_fold_path, 'data', 'antigen_info_SI_interaction')
output_dir = os.path.join(new_fold_path, 'data', 'all_antigen_info_SI')
os.makedirs(output_dir, exist_ok=True)

df_combined = []
# Process each TSV file in the input directory
for tsv_filename in os.listdir(input_dir):
    if tsv_filename.endswith('.tsv'):
        file_path = os.path.join(input_dir, tsv_filename)
        df = pd.read_csv(file_path, sep='\t')

        ab_cols = ['G', 'A', 'V', 'I', 'L', 'S', 'T', 'D', 'N',
           'E', 'Q', 'K', 'R', 'C', 'M', 'F', 'Y', 'W', 'H', 'P']
        df['Status'] = np.where(
        df[ab_cols].sum(axis=1) == 0,
        'Non-Interacting',
        'Interacting'
        )

        # Convert three-letter AA codes to one-letter sequence
        sequence = ''.join(df['AA'].map(aa_map))

        # Get the embeddings for the sequence
        embeddings = get_esm2_embedding(sequence, esm2_model, alphabet)

        embedding_df = pd.DataFrame(embeddings, columns=[f'embedding_{i}' for i in range(embeddings.shape[1])])
        df = pd.concat([df, embedding_df], axis=1)

        df_combined.append(df)

        output_file = os.path.join(output_dir, f"{tsv_filename}_with_embeddings.tsv")
        df.to_csv(output_file, sep='\t', index=False)

df_combined_all = pd.concat(df_combined, ignore_index=True)

In [ ]:
df_combined_all.columns[:50]

In [ ]:
df_combined_all

In [ ]:
df_combined_all[df_combined_all['name'].isin(["7CWO_1_L_H_A"])]

In [ ]:
# Define the directory where the merged file will be saved
save_dir = os.path.join(new_fold_path, 'data')
os.makedirs(save_dir, exist_ok=True)

# Save the concatenated dataframe to a CSV file
all_data_with_si_filepath = os.path.join(save_dir, 'all_data_with_SI.csv')
df_combined_all.to_csv(all_data_with_si_filepath, index=False)
print(f"All data has been merged and saved to {all_data_with_si_filepath}")

In [ ]:
df_combined_all